<a href="https://colab.research.google.com/github/adrian-webstep/datalab-chandra-colab/blob/main/datalab_chandra_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "chandra-ocr==0.1.8"

In [10]:
from transformers import Qwen3VLForConditionalGeneration
from transformers import Qwen3VLProcessor
from chandra.settings import settings
from pathlib import Path
from typing import List
from chandra.model.schema import BatchInputItem
from chandra.model.schema import BatchOutputItem
from chandra.model.hf import generate_hf
from chandra.output import parse_markdown
from chandra.output import parse_html
from chandra.output import parse_chunks
from chandra.output import extract_images
from chandra.input import load_image
from chandra.input import load_pdf_images
from chandra.input import parse_range_str
from chandra.output import parse_layout
from chandra.util import draw_layout
from PIL import Image

import torch
import json
import os
import requests

In [ ]:
def cuda_available():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")

In [ ]:
cuda_available()

In [ ]:
parent_path = "/content/sample_data"
PDFS_FOLDER = "./pdfs"
pdfs_folder_path = f"{parent_path}/{PDFS_FOLDER}"

def download_BERT():
    PDFs = [
        # {'title': "Attention Is All You Need", 'file': "https://arxiv.org/pdf/1706.03762"},
        # {'title': "Deep Residual Learning", 'file': "https://arxiv.org/pdf/1512.03385"},
        {'title': "BERT", 'file': "https://arxiv.org/pdf/1810.04805"},
        # {'title': "GPT-3", 'file': "https://arxiv.org/pdf/2005.14165"},
        # {'title': "Adam Optimizer", 'file': "https://arxiv.org/pdf/1412.6980"},
        # {'title': "GANs", 'file': "https://arxiv.org/pdf/1406.2661"},
        # {'title': "U-Net", 'file': "https://arxiv.org/pdf/1505.04597"},
        # {'title': "DALL-E 2", 'file': "https://arxiv.org/pdf/2204.06125"},
        # {'title': "Stable Diffusion", 'file': "https://arxiv.org/pdf/2112.10752"}
    ]

    # Add folder to save PDFs
    folder_path = PDFS_FOLDER
    os.makedirs(folder_path, exist_ok=True)

    for pdf in PDFs:
        file_path = f"{folder_path}/{pdf['title']}.pdf"
        # Skip if file already exists
        if os.path.exists(file_path):
            print(f"Skipping {pdf['title']} because it already exists")
            continue
        with open(file_path, 'wb') as f:
            f.write(requests.get(pdf['file']).content)
        print("saved paper for", pdf['title'])

In [ ]:


def load_model():
    """
    Modifisert kode av `from chandra.model.hf import load_model`
    """
    setting_torch_device = settings.TORCH_DEVICE
    setting_torch_dtype = settings.TORCH_DTYPE
    setting_torch_attn = settings.TORCH_ATTN
    setting_model_checkpoint = settings.MODEL_CHECKPOINT
    setting_local_model_checkpoint = "./models/chandra_model"
    setting_local_processor_checkpoint = "./models/chandra_processor"

    device_map = "auto"
    if setting_torch_device:
        device_map = {"": setting_torch_device}

    kwargs = {
        "dtype": setting_torch_dtype,
        "device_map": device_map,
    }
    if setting_torch_attn:
        kwargs["attn_implementation"] = setting_torch_attn

    # if local checkpoint exists, load from local
    if Path(setting_local_model_checkpoint).exists():
        print("Loading model from local checkpoint")
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            setting_local_model_checkpoint, **kwargs
        )
    else:
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            setting_model_checkpoint, **kwargs
        )
        model.save_pretrained(setting_local_model_checkpoint)
    model = model.eval()

    # if processor checkpoint is local, load from local
    if Path(setting_local_processor_checkpoint).exists():
        print("Loading processor from local checkpoint")
        processor = Qwen3VLProcessor.from_pretrained(setting_local_processor_checkpoint)
        processor.save_pretrained(setting_local_processor_checkpoint)
    else:
        processor = Qwen3VLProcessor.from_pretrained(setting_model_checkpoint)

    model.processor = processor

    return model


class InferenceManager:
    """
    Modifisert kode av `from chandra.model import InferenceManager`
    """
    def __init__(self):
        self.model = load_model()

    def generate(
        self, batch: List[BatchInputItem], max_output_tokens=None, **kwargs
    ) -> List[BatchOutputItem]:
        output_kwargs = {}
        if "include_images" in kwargs:
            output_kwargs["include_images"] = kwargs.pop("include_images")
        if "include_headers_footers" in kwargs:
            output_kwargs["include_headers_footers"] = kwargs.pop(
                "include_headers_footers"
            )

        results = generate_hf(
            batch, self.model, max_output_tokens=max_output_tokens, **kwargs
        )

        output = []
        for result, input_item in zip(results, batch):
            chunks = parse_chunks(result.raw, input_item.image)
            output.append(
                BatchOutputItem(
                    markdown=parse_markdown(result.raw, **output_kwargs),
                    html=parse_html(result.raw, **output_kwargs),
                    chunks=chunks,
                    raw=result.raw,
                    page_box=[0, 0, input_item.image.width, input_item.image.height],
                    token_count=result.token_count,
                    images=extract_images(result.raw, chunks, input_item.image),
                    error=result.error,
                )
            )
        return output


def run_ocr_on_image(
    image: Image.Image,
    model: InferenceManager,
    output_dir: Path,
    image_name: str,
    save_layout: bool = True,
    save_images: bool = True,
) -> dict:
    """
    Run OCR on a single image and save results.

    Args:
        image: PIL Image object
        model: InferenceManager instance with HF method
        output_dir: Directory to save results
        image_name: Base name for output files
        save_layout: Whether to save layout visualization
        save_images: Whether to save extracted images

    Returns:
        Dictionary with results metadata
    """
    print(f"Processing {image_name}...")

    # Create batch item for OCR
    batch_item = BatchInputItem(
        image=image,
        prompt_type="ocr_layout",
    )

    # Run inference
    result = model.generate([batch_item])[0]

    # Parse results
    layout = parse_layout(result.raw, image)
    markdown = result.markdown
    html = result.html
    chunks = result.chunks

    # Save markdown
    md_path = output_dir / f"{image_name}.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(markdown)
    print(f"  ✓ Saved markdown: {md_path}")

    # Save HTML
    html_path = output_dir / f"{image_name}.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"  ✓ Saved HTML: {html_path}")

    # Save JSON with layout and chunks
    json_path = output_dir / f"{image_name}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2)
    print(f"  ✓ Saved layout JSON: {json_path}")

    # Save layout visualization
    if save_layout:
        layout_image = draw_layout(image, layout)
        layout_path = output_dir / f"{image_name}_layout.png"
        layout_image.save(layout_path)
        print(f"  ✓ Saved layout image: {layout_path}")

    # Save extracted images
    if save_images and result.images:
        images_subdir = output_dir / f"{image_name}_images"
        images_subdir.mkdir(exist_ok=True)
        for img_name, img in result.images.items():
            img_path = images_subdir / img_name
            img.save(img_path)
        print(f"  ✓ Saved {len(result.images)} extracted images to {images_subdir}")

    # Save raw output
    raw_path = output_dir / f"{image_name}_raw.html"
    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(result.raw)
    print(f"  ✓ Saved raw output: {raw_path}")

    return {
        "file": image_name,
        "token_count": result.token_count,
        "chunks": len(chunks),
        "images_extracted": len(result.images),
        "error": result.error,
    }

def pdf_ocr(input_file:str, output_file:str="./output", page_range:str=None, save_layout:bool=True, save_images:bool=True):
    # Validate input file
    input_path = Path(input_file)
    if not input_path.exists():
        print(f"Error: Input file not found: {input_path}")
        return 1

    # Create output directory
    output_dir = Path(output_file)
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {output_dir}\n")

    # Load model
    print("Loading HuggingFace model...")
    model = InferenceManager()
    print("Model loaded.\n")

    # Load images
    print(f"Loading image(s) from: {input_path}")
    if input_path.suffix.lower() == ".pdf":
        page_range_to_use = None
        if page_range:
            page_range_to_use = parse_range_str(page_range)
        images = load_pdf_images(str(input_path), page_range_to_use or [])
        print(f"Loaded {len(images)} page(s) from PDF\n")
    else:
        images = [load_image(str(input_path))]
        print("Loaded image\n")

    # Process each image
    results = []
    for i, image in enumerate(images):
        base_name = input_path.stem
        if len(images) > 1:
            image_name = f"{base_name}_page{i}"
        else:
            image_name = base_name

        result = run_ocr_on_image(
            image,
            model,
            output_dir,
            image_name,
            save_layout=save_layout,
            save_images=save_images,
        )
        results.append(result)
        print()



def run_chandra_ocr():
    cuda_available()
    pdf_path = f"{PDFS_FOLDER}/BERT.pdf"
    if not Path(pdf_path).exists():
        download_BERT()
    pdf_ocr(input_file=pdf_path, output_file="./output", page_range="1", save_layout=True, save_images=True)
